<a href="https://colab.research.google.com/github/Midushii/AI_vs_Human_ChildMedia/blob/main/Human_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q opencv-python-headless librosa pyloudnorm scenedetect soundfile pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.2/146.2 kB 6.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DRIVE_PATH = "/content/drive/MyDrive/ai generated kids content"
HUMAN_FOLDER = os.path.join(BASE_DRIVE_PATH, "Human_Content")

OUTPUT_DIR = os.path.join(BASE_DRIVE_PATH, "features_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

HUMAN_CSV = os.path.join(OUTPUT_DIR, "human_features_v3.csv")   # new filename, don't overwrite v2

VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov"}
FRAME_SKIP = 5

Mounted at /content/drive


In [ ]:
print("Human folder exists:", os.path.isdir(HUMAN_FOLDER), "->", HUMAN_FOLDER)
if os.path.isdir(HUMAN_FOLDER):
    files = [f for f in os.listdir(HUMAN_FOLDER) if os.path.splitext(f)[1].lower() in VIDEO_EXTS]
    print(f"{len(files)} video files found")
    print("Sample:", files[:3])

Human folder exists: True -> /content/drive/MyDrive/ai generated kids content/Human_Content
267 video files found
Sample: ['A-rEb0KuopI_In The Fall.mp4', 'jsSGt9YgEoA_Mr. Bean Cartoon Flipbook #9 ｜ Scared Bean Flip Book ｜ Flip Book Artist 2020.mp4', 'H1cCU62loqU_BENCH - STOP MOTION ANIMATED SHORT FILM #animation #waaber #bench.mp4']


In [ ]:
import cv2
import numpy as np
import pandas as pd
import librosa
import pyloudnorm as pyln
import soundfile as sf
import subprocess
import tempfile
from pathlib import Path
from tqdm import tqdm

from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector

/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


In [ ]:
def get_video_id(filename):
    """First 11 characters = the fixed-length YouTube video ID, present in
    every filename regardless of separator/title/special characters."""
    stem = Path(filename).stem
    return stem[:11]

In [ ]:
def extract_pacing_features(video_path):
    for backend in ['opencv', 'pyav']:
        try:
            video = open_video(str(video_path), backend=backend)
            scene_manager = SceneManager()
            scene_manager.add_detector(ContentDetector(threshold=27.0))
            scene_manager.detect_scenes(video)
            scene_list = scene_manager.get_scene_list()

            duration_sec = video.duration.get_seconds()
            num_cuts = len(scene_list)

            if duration_sec <= 0:
                continue

            shot_count_per_sec = num_cuts / duration_sec
            avg_shot_duration_sec = duration_sec / num_cuts if num_cuts > 0 else duration_sec

            return {
                "shot_count_per_sec": round(shot_count_per_sec, 4),
                "avg_shot_duration_sec": round(avg_shot_duration_sec, 3),
                "num_cuts": num_cuts,
                "duration_sec": round(duration_sec, 3),
            }
        except Exception as e:
            print(f"    [pacing: backend '{backend}' failed for {video_path.name}: {e}]")
            continue

    print(f"    [pacing: both backends failed for {video_path.name}]")
    return {"shot_count_per_sec": None, "avg_shot_duration_sec": None,
            "num_cuts": None, "duration_sec": None}

In [ ]:
def extract_visual_features(video_path, frame_skip=FRAME_SKIP):
    cap = cv2.VideoCapture(str(video_path), cv2.CAP_FFMPEG)
    if not cap.isOpened():
        print(f"    [visual: could not open {video_path.name}]")
        return {"brightness_mean": None, "saturation_mean": None,
                "color_warmness_pct": None, "visual_complexity": None}

    brightness_vals, sat_vals, warm_vals, complexity_vals = [], [], [], []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_skip == 0:
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

            brightness_vals.append(np.mean(v))
            sat_vals.append(np.mean(s))

            colorful_mask = (s > 40) & (v > 40)
            if np.any(colorful_mask):
                warm_mask = colorful_mask & ((h <= 30) | (h >= 150))
                warm_vals.append(100.0 * np.sum(warm_mask) / np.sum(colorful_mask))

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 200)
            complexity_vals.append(100.0 * np.sum(edges > 0) / edges.size)

        frame_idx += 1

    cap.release()

    if not brightness_vals:
        print(f"    [visual: opened but 0 readable frames for {video_path.name}]")
        return {"brightness_mean": None, "saturation_mean": None,
                "color_warmness_pct": None, "visual_complexity": None}

    return {
        "brightness_mean": round(float(np.mean(brightness_vals)), 3),
        "saturation_mean": round(float(np.mean(sat_vals)), 3),
        "color_warmness_pct": round(float(np.mean(warm_vals)), 3) if warm_vals else None,
        "visual_complexity": round(float(np.mean(complexity_vals)), 3),
    }

In [ ]:
def extract_audio_to_wav(video_path, out_sr=48000):
    tmp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-ac", "1", "-ar", str(out_sr),
           "-vn", "-loglevel", "error", tmp_wav]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0 or not os.path.exists(tmp_wav):
        raise RuntimeError(f"ffmpeg failed: {result.stderr.decode(errors='ignore')[:200]}")
    return tmp_wav


def extract_audio_features(video_path):
    result = {"loudness": None, "loudness_method": None, "tempo_bpm": None, "sound_brightness": None}
    tmp_wav = None
    try:
        tmp_wav = extract_audio_to_wav(video_path)
        y, sr = sf.read(tmp_wav)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if y.size == 0:
            return result

        try:
            meter = pyln.Meter(sr)
            result["loudness"] = round(float(meter.integrated_loudness(y)), 3)
            result["loudness_method"] = "LUFS"
        except Exception as e:
            rms = np.sqrt(np.mean(y**2))
            result["loudness"] = round(float(20 * np.log10(rms + 1e-9)), 3)
            result["loudness_method"] = "RMS_dBFS_fallback"
            print(f"    [pyloudnorm fallback for {video_path.name}: {e}]")

        try:
            tempo, _ = librosa.beat.beat_track(y=y.astype(np.float32), sr=sr)
            result["tempo_bpm"] = round(float(tempo), 2)
        except Exception as e:
            print(f"    [tempo failed for {video_path.name}: {e}]")

        try:
            centroid = librosa.feature.spectral_centroid(y=y.astype(np.float32), sr=sr)
            result["sound_brightness"] = round(float(np.mean(centroid)), 2)
        except Exception as e:
            print(f"    [spectral centroid failed for {video_path.name}: {e}]")

    except Exception as e:
        print(f"    [audio extraction failed for {video_path.name}: {e}]")
    finally:
        if tmp_wav and os.path.exists(tmp_wav):
            os.remove(tmp_wav)

    return result

In [ ]:
def _save_rows(rows, output_csv):
    if not rows:
        return
    new_df = pd.DataFrame(rows)
    if os.path.exists(output_csv):
        old_df = pd.read_csv(output_csv)
        combined = pd.concat([old_df, new_df], ignore_index=True).drop_duplicates(subset="video_id", keep="last")
    else:
        combined = new_df
    combined.to_csv(output_csv, index=False)


def process_folder(folder_path, output_csv, label):
    folder_path = Path(folder_path)
    video_files = [p for p in folder_path.iterdir() if p.suffix.lower() in VIDEO_EXTS]
    print(f"Found {len(video_files)} videos in {folder_path}")

    already_done = set()
    if os.path.exists(output_csv):
        already_done = set(pd.read_csv(output_csv)["video_id"].astype(str))
        print(f"  {len(already_done)} already processed -- skipping those")

    rows = []
    for video_path in tqdm(video_files, desc=folder_path.name):
        vid = get_video_id(video_path.name)
        if vid in already_done:
            continue

        row = {"video_id": vid, "filename": video_path.name, "label": label}
        try:
            row.update(extract_pacing_features(video_path))
        except Exception as e:
            print(f"  [pacing error: {e}]")
        try:
            row.update(extract_visual_features(video_path))
        except Exception as e:
            print(f"  [visual error: {e}]")
        try:
            row.update(extract_audio_features(video_path))
        except Exception as e:
            print(f"  [audio error: {e}]")

        rows.append(row)
        if len(rows) % 10 == 0:
            _save_rows(rows, output_csv)

    _save_rows(rows, output_csv)
    print(f"Done -- saved to {output_csv}")

In [ ]:
process_folder(HUMAN_FOLDER, HUMAN_CSV, label="human")

Found 267 videos in /content/drive/MyDrive/ai generated kids content/Human_Content


Human_Content:   0%|          | 0/267 [00:00<?, ?it/s]INFO:pyscenedetect:Detecting scenes...
  duration_sec = video.duration.get_seconds()



    [visual: opened but 0 readable frames for A-rEb0KuopI_In The Fall.mp4]


  result["tempo_bpm"] = round(float(tempo), 2)

Human_Content:   0%|          | 1/267 [00:37<2:47:18, 37.74s/it]INFO:pyscenedetect:Detecting scenes...
  duration_sec = video.duration.get_seconds()



    [visual: opened but 0 readable frames for jsSGt9YgEoA_Mr. Bean Cartoon Flipbook #9 ｜ Scared Bean Flip Book ｜ Flip Book Artist 2020.mp4]


...

    [visual: opened but 0 readable frames for 3FSM-Kvc88A - Trio of the Ocean ｜ Sing Along with Baby Shark ｜ Pinkfong Songs for Children.mp4]


  result["tempo_bpm"] = round(float(tempo), 2)

Human_Content:  99%|█████████▉| 264/267 [40:36<00:14,  4.82s/it]INFO:pyscenedetect:Detecting scenes...
  duration_sec = video.duration.get_seconds()

  result["tempo_bpm"] = round(float(tempo), 2)

Human_Content:  99%|█████████▉| 265/267 [41:42<00:46, 23.09s/it]INFO:pyscenedetect:Detecting scenes...
  duration_sec = video.duration.get_seconds()



    [visual: opened but 0 readable frames for 3IKWn5oEN34 - Ni Hao Panda ｜ Panda ｜ Animal Songs ｜ Pinkfong Songs for Children.mp4]


  result["tempo_bpm"] = round(float(tempo), 2)

Human_Content: 100%|█████████▉| 266/267 [41:46<00:17, 17.49s/it]INFO:pyscenedetect:Detecting scenes...
  duration_sec = video.duration.get_seconds()

  result["tempo_bpm"] = round(float(tempo), 2)

Human_Content: 100%|██████████| 267/267 [42:57<00:00,  9.65s/it]

Done -- saved to /content/drive/MyDrive/ai generated kids content/features_output/human_features_v3.csv
